#Statement of Intent
My goal is to build an end-to-end ETL pipeline in PySpark to process player performance data and generate predictive features. Trained and compared multiple ML models (Random Forest, Gradient Boosting, Logistic Regression) to predict match outcomes. Later model versions will integrate MLflow for model tracking and cross-validation to evaluate performance. Designed for deployment to support real-time match predictions or possibly used to bet for the moneyline, though the latter is not the desired purpose.

#Data Sourcing
The data is sourced from match statistics provided by Jeff Sackmann.

The last 4 years worth of matches from Jeff's GitHub repo have been stored in s3 on a free AWS account. I am starting the feature engineering by initiating the spark session and pulling the raw files from the s3 bucket.

In [0]:
from pyspark.sql import (
    SparkSession,
    types,
    functions as F,
)

from pyspark.sql.window import Window


import mlflow
import mlflow.spark

from mlflow.models.signature import infer_signature

import os
import pandas as pd

I convert the files to delta for faster pulling, version control, and updating records if needed. I only need to run that cell once.

In [0]:
#Convert to delta
df = spark.read.csv("s3://data-storage-for-projects/Tennis Analytics Project/v1/raw/"
                    , header=True, inferSchema=True)
df.write.format("delta").mode("overwrite").save("s3://data-storage-for-projects/Tennis Analytics Project/v1/delta/")

In [0]:
#Bring data into notebook
df = spark.read.format("delta").load("s3://data-storage-for-projects/Tennis Analytics Project/v1/delta/")
display(df)

In [0]:
#This function will be used later to evaluate each model through a variety of metrics
from pyspark.ml.evaluation import BinaryClassificationEvaluator, MulticlassClassificationEvaluator
from pyspark.mllib.evaluation import BinaryClassificationMetrics, MulticlassMetrics

def evaluate_model(predictions):
    # ==========================================
    # 1. AUC Metrics (BinaryClassificationEvaluator)
    # ==========================================
    auc_roc_evaluator = BinaryClassificationEvaluator(labelCol='id_a_won', metricName='areaUnderROC')
    auc_pr_evaluator = BinaryClassificationEvaluator(labelCol='id_a_won', metricName='areaUnderPR')

    auc_roc = auc_roc_evaluator.evaluate(predictions)
    auc_pr = auc_pr_evaluator.evaluate(predictions)

    # ==========================================
    # 2. Accuracy, Precision, Recall, F1 (MulticlassClassificationEvaluator)
    # ==========================================
    accuracy_evaluator = MulticlassClassificationEvaluator(labelCol='id_a_won', predictionCol='prediction', metricName='accuracy')
    precision_evaluator = MulticlassClassificationEvaluator(labelCol='id_a_won', predictionCol='prediction', metricName='weightedPrecision')
    recall_evaluator = MulticlassClassificationEvaluator(labelCol='id_a_won', predictionCol='prediction', metricName='weightedRecall')
    f1_evaluator = MulticlassClassificationEvaluator(labelCol='id_a_won', predictionCol='prediction', metricName='f1')

    accuracy = accuracy_evaluator.evaluate(predictions)
    precision = precision_evaluator.evaluate(predictions)
    recall = recall_evaluator.evaluate(predictions)
    f1 = f1_evaluator.evaluate(predictions)

    # ==========================================
    # 3. Confusion Matrix (manual calculation)
    # ==========================================
    predictions.groupBy('id_a_won', 'prediction').count().show()

    # Or create it properly:
    tp = predictions.filter((F.col('id_a_won') == 1) & (F.col('prediction') == 1)).count()
    tn = predictions.filter((F.col('id_a_won') == 0) & (F.col('prediction') == 0)).count()
    fp = predictions.filter((F.col('id_a_won') == 0) & (F.col('prediction') == 1)).count()
    fn = predictions.filter((F.col('id_a_won') == 1) & (F.col('prediction') == 0)).count()

    print("=" * 60)
    print(f"{model_name.upper()} - EVALUATION METRICS")
    print("=" * 60)
    print(f"AUC-ROC:              {auc_roc:.4f}")
    print(f"AUC-PR:               {auc_pr:.4f}")
    print(f"Accuracy:             {accuracy:.4f}")
    print(f"Precision (Weighted): {precision:.4f}")
    print(f"Recall (Weighted):    {recall:.4f}")
    print(f"F1 Score:             {f1:.4f}")
    print("=" * 60)
    print("CONFUSION MATRIX")
    print("=" * 60)
    print(f"True Positives:       {tp:,}")
    print(f"True Negatives:       {tn:,}")
    print(f"False Positives:      {fp:,}")
    print(f"False Negatives:      {fn:,}")
    print("=" * 60)

    return {
        'auc_roc': auc_roc,
        'auc_pr': auc_pr,
        'accuracy': accuracy,
        'precision': precision,
        'recall': recall,
        'f1': f1,
        'confusion_matrix': {'tp': tp, 'tn': tn, 'fp': fp, 'fn': fn}
    }

#Baseline Models aka V1
For my baseline models, I am using tournament conditions such as surface, tournament level, best of, and round of the match. I will also include rankings of each player, but I will need to have two rows per match so that each player is player 1. The data in its current form has winners all in one column and could train the model incorrectly. As for the types of models, I plan to create and compare three--logistic regression, random forest, and gradient boosting.

In [0]:
#Columns for player a and player b rank, age, and hand. It also says if a won. Two are used to prevent data leakage.
d1 = df.select('surface', 'tourney_level', 'round',
        'best_of', 'winner_hand', 'loser_hand',
        F.col('winner_id').alias('id_a'), F.col('loser_id').alias('id_b'), 
        F.col('winner_rank').alias('id_a_rank'), F.col('loser_rank').alias('id_b_rank'),
        F.col('winner_age').alias('id_a_age'), F.col('loser_age').alias('id_b_age')
        ).withColumn('rank_diff', F.col('id_a_rank') - F.col('id_b_rank')) \
        .withColumn('best_of', F.when(F.col('best_of')==3, 0).otherwise(1)) \
        .withColumn('id_a_hand', F.when(F.col('winner_hand')=='R', 0).otherwise(1)) \
        .withColumn('id_b_hand', F.when(F.col('loser_hand')=='R', 0).otherwise(1))  \
        .withColumn("round_encoded",
        F.when(df.round == "RR", 0)
        .when(df.round == "R128", 1)
        .when(df.round == "R64", 2)
        .when(df.round == "R32", 3)
        .when(df.round == "R16", 4)
        .when(df.round == "QF", 5)
        .when(df.round == "SF", 6)
        .when(df.round == "BR", 6.5)
        .when(df.round == "F", 7)
        .otherwise(None)
        ) \
        .withColumn('id_a_won', F.lit(1))


d2 = df.select('surface', 'tourney_level', 'round',
        'best_of', 'winner_hand', 'loser_hand',
        F.col('loser_id').alias('id_a'), F.col('winner_id').alias('id_b'), 
        F.col('loser_rank').alias('id_a_rank'), F.col('winner_rank').alias('id_b_rank'),
        F.col('loser_age').alias('id_a_age'), F.col('winner_age').alias('id_b_age')
        ).withColumn('rank_diff', F.col('id_a_rank') - F.col('id_b_rank')) \
        .withColumn('best_of', F.when(F.col('best_of')==3, 0).otherwise(1)) \
        .withColumn('id_a_hand', F.when(F.col('loser_hand')=='R', 0).otherwise(1)) \
        .withColumn('id_b_hand', F.when(F.col('winner_hand')=='R', 0).otherwise(1)) \
        .withColumn("round_encoded",
        F.when(F.col('round') == "RR", 0)
        .when(F.col('round') == "R128", 1)
        .when(F.col('round') == "R64", 2)
        .when(F.col('round') == "R32", 3)
        .when(F.col('round') == "R16", 4)
        .when(F.col('round') == "QF", 5)
        .when(F.col('round') == "SF", 6)
        .when(F.col('round') == "BR", 6.5)
        .when(F.col('round') == "F", 7)
        .otherwise(None)
        ) \
        .withColumn('id_a_won', F.lit(0))

d = d1.union(d2).drop('winner_hand', 'loser_hand', 'round') \
        .filter(F.col('surface').isNotNull() & F.col('rank_diff').isNotNull()
                & F.col('id_a_age').isNotNull() & F.col('id_b_age').isNotNull()) 
display(d)

In [0]:
#Create ML Pipeline starting with training data, string indexer and one hot encoder for logisitic regression
from pyspark.ml import Pipeline
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler

#Traininng and testing data
train_df, test_df = d.randomSplit([.8, .2], seed=42)

#Indexing and Encoding
indexer_surface = StringIndexer(inputCol='surface', outputCol='surface_index')
encoder_surface = OneHotEncoder(inputCol='surface_index', outputCol='surface_vec')

indexer_tourney = StringIndexer(inputCol='tourney_level', outputCol='tourney_index')
encoder_tourney = OneHotEncoder(inputCol='tourney_index', outputCol='tourney_vec')

#X variables
feature_cols = ['surface_vec', 'tourney_vec', 'round_encoded', 'best_of', 'id_a_rank',
                'id_b_rank','id_a_age','id_b_age','rank_diff','id_a_hand','id_b_hand']

assembler = VectorAssembler(inputCols=feature_cols, outputCol='features')

In [0]:
#Logistic Regression Model
from pyspark.ml.classification import LogisticRegression

lr = LogisticRegression(featuresCol='features', labelCol='id_a_won')

pipeline_lr = Pipeline(stages = [
    indexer_surface, encoder_surface,
    indexer_tourney, encoder_tourney,
    assembler,
    lr
])

#Create Model and save to s3
lr_model = pipeline_lr.fit(train_df)
lr_model.save("s3://data-storage-for-projects/Tennis Analytics Project/v1/v1-models/logistic-regressions/1.0/")

#Generate Predictions and Evaluate Accuracy
lr_predict = lr_model.transform(test_df)
lr_eval = evaluate_model(lr_predict)

#Remove from Databricks Memory to create more models
del lr_model, lr_predict

In [0]:
#Random Forest
from pyspark.ml.classification import RandomForestClassifier

rf = RandomForestClassifier(featuresCol='features', labelCol='id_a_won', numTrees=50, maxDepth=10)

feature_cols = ['surface_index', 'tourney_index', 'round_encoded', 'best_of', 'id_a_rank',
                'id_b_rank','id_a_age','id_b_age','rank_diff','id_a_hand','id_b_hand']

assembler = VectorAssembler(inputCols=feature_cols, outputCol='features')

pipeline_rf = Pipeline(stages = [
    indexer_surface,
    indexer_tourney,
    assembler,
    rf
])

#Create Model and save to s3
rf_model = pipeline_rf.fit(train_df)
rf_model.save("s3://data-storage-for-projects/Tennis Analytics Project/v1/v1-models/random-forests/1.1/")

#Generate Predictions and Evaluate Accuracy
rf_predict = rf_model.transform(test_df)
rf_eval = evaluate_model(rf_predict)

#Remove from Databricks Memory to create more models
del rf_model, rf_predict

In [0]:
#Gradient Boosting
from pyspark.ml.classification import GBTClassifier

gb = GBTClassifier(labelCol="id_a_won", featuresCol="features", maxDepth=6, maxIter=20)

feature_cols = ['surface_index', 'tourney_index', 'round_encoded', 'best_of', 'id_a_rank',
                'id_b_rank','id_a_age','id_b_age','rank_diff','id_a_hand','id_b_hand']

assembler = VectorAssembler(inputCols=feature_cols, outputCol='features')

pipeline_gb = Pipeline(stages = [
    indexer_surface,
    indexer_tourney,
    assembler,
    gb
])

#Create Model and save to s3
gb_model = pipeline_gb.fit(train_df)
gb_model.save("s3://data-storage-for-projects/Tennis Analytics Project/v1/v1-models/gradient-boostings/1.0/")

#Generate Predictions and Evaluate Accuracy
gb_predict = gb_model.transform(test_df)
gb_eval = evaluate_model(gb_predict)

#Remove from Databricks Memory to create more models
del gb_model, gb_predict

The evaluations for the three models are shown below.
| **Metric**             | **Logistic Regression** | **Random Forest** | **Gradient Boosting** |
|--------------------------|--------------------------|--------------------|------------------------|
| **AUC-ROC**              | 0.6698                   | 0.6953             | 0.6967                 |
| **AUC-PR**               | 0.6824                   | 0.7056             | 0.7066                 |
| **Accuracy**             | 0.6178                   | 0.6283             | 0.6338                 |
| **Precision (Weighted)** | 0.6213                   | 0.6299             | 0.6358                 |
| **Recall (Weighted)**    | 0.6178                   | 0.6283             | 0.6338                 |
| **F1 Score**             | 0.6175                   | 0.6285             | 0.6339                 |
| **True Positives**       | 1,386                    | 1,467              | 1,466                  |
| **True Negatives**       | 1,436                    | 1,403              | 1,429                  |
| **False Positives**      | 743                      | 776                | 750                    |
| **False Negatives**      | 1,003                    | 922                | 923                    |

Gradient Boosting appears to be the best v1 model, evidenced by the higher scores in AUC-ROC and F1. Random Forest is not far behind, and the results are all higher than 50%, suggesting they are better than chance. However, there is room for improvement.

#V2
For this round of models, there are several factors that I would like to add to increase the evaluation scores. The most obvious is to add more data. Instead of four years let's see what six or eight years provides. A time series cross validation will be implemented instead of one random 80/20 split. I would also like to generate running totals of stats before the match is counted. This would include win percentage, 1st serve percentage, and break percentage for the last ten matches. Using the statistics for the match being predicted would result in data leakage, so I cannot use those until I factor them in for the next match.

In [0]:
#Convert to delta
df = spark.read.csv("s3://data-storage-for-projects/Tennis Analytics Project/v2/raw/"
                    , header=True, inferSchema=True)
df.write.format("delta").mode("overwrite").save("s3://data-storage-for-projects/Tennis Analytics Project/v2/delta/")

In [0]:
#Bring data into notebook
df = spark.read.format("delta").load("s3://data-storage-for-projects/Tennis Analytics Project/v1/delta/")
display(df)

In [0]:
df = df.withColumn('tourney_date', F.to_date(F.col('tourney_date').cast('string'), 'yyyyMMdd')) \
    .withColumn("id", F.monotonically_increasing_id()) \
    .withColumn("round_encoded",
        F.when(F.col('round') == "RR", 0)
        .when(F.col('round') == "R128", 1)
        .when(F.col('round') == "R64", 2)
        .when(F.col('round') == "R32", 3)
        .when(F.col('round') == "R16", 4)
        .when(F.col('round') == "QF", 5)
        .when(F.col('round') == "SF", 6)
        .when(F.col('round') == "BR", 6.5)
        .when(F.col('round') == "F", 7)
        .otherwise(None))

df1 = df.select(F.col('winner_id').alias('player_id'), 'id', 'tourney_date', 'round_encoded', 'minutes', F.col('w_df').alias('df'), 
                F.col('w_bpFaced').alias('bp_faced'), F.col('l_bpFaced').alias('bp_created'),
                'w_1stIn', 'w_svpt', 'w_1stWon', 'w_2ndWon', 'w_bpSaved',
                'l_bpSaved', 'l_svpt', 'l_1stWon', 'l_2ndWon', 'l_SvGms'
 ) \
    .withColumn('first_pct', F.try_divide(F.col('w_1stIn'), F.col('w_svpt'))) \
    .withColumn('first_win_pct', F.try_divide(F.col('w_1stWon'), F.col('w_1stIn'))) \
    .withColumn('second_win_pct', F.try_divide(F.col('w_2ndWon'), F.col('w_svpt') - F.col('w_1stIn'))) \
    .withColumn('bp_saved_pct', F.try_divide(F.col('w_bpSaved'), F.col('bp_faced'))) \
    .withColumn('bp_convert_pct', F.try_divide(F.col('bp_created') - F.col('l_bpSaved'), F.col('bp_created'))) \
    .withColumn('return_pt_win_pct', F.try_divide(F.col('l_svpt') - F.col('l_1stWon') - F.col('l_2ndWon'), F.col('l_svpt'))) \
    .withColumn('break_pct', F.try_divide(F.col('bp_created') - F.col('l_bpSaved'), F.col('l_SvGms'))) \
    .drop('w_1stIn', 'w_svpt', 'w_1stWon', 'w_2ndWon', 'w_bpSaved', 'l_bpSaved', 'l_svpt', 'l_1stWon', 'l_2ndWon', 'l_SvGms')

df2 = (df.select(F.col('loser_id').alias('player_id'), 'id', 'tourney_date', 'round_encoded', 'minutes', F.col('l_df').alias('df'), 
                F.col('l_bpFaced').alias('bp_faced'), F.col('w_bpFaced').alias('bp_created'),
                'l_1stIn', 'l_svpt', 'l_1stWon', 'l_2ndWon', 'l_bpSaved',
                'w_bpSaved', 'w_svpt', 'w_1stWon', 'w_2ndWon', 'w_SvGms'
 ) \
    .withColumn('first_pct', F.try_divide(F.col('l_1stIn'), F.col('l_svpt'))) \
    .withColumn('first_win_pct', F.try_divide(F.col('l_1stWon'), F.col('l_1stIn'))) \
    .withColumn('second_win_pct', F.try_divide(F.col('l_2ndWon'), F.col('l_svpt') - F.col('l_1stIn'))) \
    .withColumn('bp_saved_pct', F.try_divide(F.col('l_bpSaved'), F.col('bp_faced'))) \
    .withColumn('bp_convert_pct', F.try_divide(F.col('bp_created') - F.col('w_bpSaved'), F.col('bp_created'))) \
    .withColumn('return_pt_win_pct', F.try_divide(F.col('w_svpt') - F.col('w_1stWon') - F.col('w_2ndWon'), F.col('w_svpt'))) \
    .withColumn('break_pct', F.try_divide(F.col('bp_created') - F.col('w_bpSaved'), F.col('w_SvGms'))) \
    .drop('l_1stIn', 'l_svpt', 'l_1stWon', 'l_2ndWon', 'l_bpSaved', 'w_bpSaved', 'w_svpt', 'w_1stWon', 'w_2ndWon', 'w_SvGms'))

player = df1.union(df2)
display(player)

In [0]:
#Stats for last 5 matches
w = Window.partitionBy('player_id').orderBy('tourney_date', 'round_encoded').rowsBetween(-5, -1)

player = player.withColumn('first_pct_avg', F.avg('first_pct').over(w)) \
    .withColumn('first_win_pct_avg', F.avg('first_win_pct').over(w)) \
    .withColumn('second_win_pct_avg', F.avg('second_win_pct').over(w)) \
    .withColumn('bp_saved_pct_avg', F.avg('bp_saved_pct').over(w)) \
    .withColumn('bp_convert_pct_avg', F.avg('bp_convert_pct').over(w)) \
    .withColumn('return_pt_win_pct_avg', F.avg('return_pt_win_pct').over(w)) \
    .withColumn('break_pct_avg', F.avg('break_pct').over(w)) \
    .withColumn('df_avg', F.avg('df').over(w)) \
    .withColumn('bp_faced_avg', F.avg('bp_faced').over(w)) \
    .withColumn('bp_created_avg', F.avg('bp_created').over(w)) \
    .withColumn('minutes_sum', F.sum('minutes').over(w))

display(player)

In [0]:
d = df.join(
    player.alias('pa'),
    (df.winner_id == F.col('pa.player_id')) & (df.id == F.col('pa.id')) & (df.tourney_date == F.col('pa.tourney_date')),
    'left'
).join(
    player.alias('pb'),
    (df.loser_id == F.col('pb.player_id')) & (df.id == F.col('pb.id')) & (df.tourney_date == F.col('pa.tourney_date')),
    'left'
).select(
    # Keep df columns (not duplicated)
        df.id,
        df.tourney_date,
        df.round,
        df.surface,
        df.tourney_level,
        df.best_of,
        df.winner_id,
        df.loser_id,
        df.winner_rank,
        df.loser_rank,
        df.winner_age,
        df.loser_age,
        df.winner_hand,
        df.loser_hand,
    
    # Winner's rolling stats from 'pa'
    F.col('pa.first_pct_avg').alias('winner_first_pct_avg'),
    F.col('pa.first_win_pct_avg').alias('winner_first_win_pct_avg'),
    F.col('pa.second_win_pct_avg').alias('winner_second_win_pct_avg'),
    F.col('pa.bp_saved_pct_avg').alias('winner_bp_saved_pct_avg'),
    F.col('pa.bp_convert_pct_avg').alias('winner_bp_convert_pct_avg'),
    F.col('pa.return_pt_win_pct_avg').alias('winner_return_pt_win_pct_avg'),
    F.col('pa.break_pct_avg').alias('winner_break_pct_avg'),
    F.col('pa.df_avg').alias('winner_df_avg'),
    F.col('pa.minutes_sum').alias('winner_minutes_sum'),
    
    # Loser's rolling stats from 'pb'
    F.col('pb.first_pct_avg').alias('loser_first_pct_avg'),
    F.col('pb.first_win_pct_avg').alias('loser_first_win_pct_avg'),
    F.col('pb.second_win_pct_avg').alias('loser_second_win_pct_avg'),
    F.col('pb.bp_saved_pct_avg').alias('loser_bp_saved_pct_avg'),
    F.col('pb.bp_convert_pct_avg').alias('loser_bp_convert_pct_avg'),
    F.col('pb.return_pt_win_pct_avg').alias('loser_return_pt_win_pct_avg'),
    F.col('pb.break_pct_avg').alias('loser_break_pct_avg'),
    F.col('pb.df_avg').alias('loser_df_avg'),
    F.col('pb.minutes_sum').alias('loser_minutes_sum')
)\
.withColumn('random_flip', F.rand(seed=42))

d = d.select('*') \
.withColumn('id_a', F.when(F.col('random_flip')>= 0.5, F.col('winner_id')).otherwise(F.col('loser_id'))) \
.withColumn('id_b', F.when(F.col('random_flip')>= 0.5, F.col('loser_id')).otherwise(F.col('winner_id'))) \
.withColumn('id_a_rank', F.when(F.col('random_flip')>= 0.5, F.col('winner_rank')).otherwise(F.col('loser_rank'))) \
.withColumn('id_b_rank', F.when(F.col('random_flip')>= 0.5, F.col('loser_rank')).otherwise(F.col('winner_rank'))) \
.withColumn('id_a_age', F.when(F.col('random_flip')>= 0.5, F.col('winner_age')).otherwise(F.col('loser_age'))) \
.withColumn('id_b_age', F.when(F.col('random_flip')>= 0.5, F.col('loser_age')).otherwise(F.col('winner_age'))) \
.withColumn('rank_diff', F.col('id_a_rank') - F.col('id_b_rank')) \
.withColumn('best_of', F.when(F.col('best_of')==3, 0).otherwise(1)) \
.withColumn('id_a_hand', F.when(F.col('random_flip')>= 0.5, F.when(F.col('winner_hand')=='R', 0).otherwise(1)).otherwise(F.when(F.col('loser_hand')=='R', 0).otherwise(1))) \
.withColumn('id_b_hand', F.when(F.col('random_flip')>= 0.5, F.when(F.col('loser_hand')=='R', 0).otherwise(1)).otherwise(F.when(F.col('winner_hand')=='R', 0).otherwise(1)))  \
.withColumn('id_a_first_pct_avg', F.when(F.col('random_flip')>= 0.5, F.col('winner_first_pct_avg')).otherwise(F.col('loser_first_pct_avg'))) \
.withColumn('id_b_first_pct_avg', F.when(F.col('random_flip')>= 0.5, F.col('loser_first_pct_avg')).otherwise(F.col
('winner_first_pct_avg'))) \
.withColumn('id_a_first_win_pct_avg', F.when(F.col('random_flip')>= 0.5, F.col('winner_first_win_pct_avg')).otherwise(F.col('loser_first_win_pct_avg'))) \
.withColumn('id_b_first_win_pct_avg', F.when(F.col('random_flip')>= 0.5, F.col('loser_first_win_pct_avg')).otherwise(F.col
('winner_first_win_pct_avg'))) \
.withColumn('id_a_second_win_pct_avg', F.when(F.col('random_flip')>= 0.5, F.col('winner_second_win_pct_avg')).otherwise(F.col('loser_second_win_pct_avg'))) \
.withColumn('id_b_second_win_pct_avg', F.when(F.col('random_flip')>= 0.5, F.col('loser_second_win_pct_avg')).otherwise(F.col
('winner_second_win_pct_avg'))) \
.withColumn('id_a_bp_saved_pct_avg', F.when(F.col('random_flip')>= 0.5, F.col('winner_bp_saved_pct_avg')).otherwise(F.col('loser_bp_saved_pct_avg'))) \
.withColumn('id_b_bp_saved_pct_avg', F.when(F.col('random_flip')>= 0.5, F.col('loser_bp_saved_pct_avg')).otherwise(F.col
('winner_bp_saved_pct_avg'))) \
.withColumn('id_a_bp_convert_pct_avg', F.when(F.col('random_flip')>= 0.5, F.col('winner_bp_convert_pct_avg')).otherwise(F.col('loser_bp_convert_pct_avg'))) \
.withColumn('id_b_bp_convert_pct_avg', F.when(F.col('random_flip')>= 0.5, F.col('loser_bp_convert_pct_avg')).otherwise(F.col
('winner_bp_convert_pct_avg'))) \
.withColumn('id_a_return_pt_win_pct_avg', F.when(F.col('random_flip')>= 0.5, F.col('winner_return_pt_win_pct_avg')).otherwise(F.col('loser_return_pt_win_pct_avg'))) \
.withColumn('id_b_return_pt_win_pct_avg', F.when(F.col('random_flip')>= 0.5, F.col('loser_return_pt_win_pct_avg')).otherwise(F.col
('winner_return_pt_win_pct_avg'))) \
.withColumn('id_a_break_pct_avg', F.when(F.col('random_flip')>= 0.5, F.col('winner_break_pct_avg')).otherwise(F.col('loser_break_pct_avg'))) \
.withColumn('id_b_break_pct_avg', F.when(F.col('random_flip')>= 0.5, F.col('loser_break_pct_avg')).otherwise(F.col
('winner_break_pct_avg'))) \
.withColumn('id_a_df_avg', F.when(F.col('random_flip')>= 0.5, F.col('winner_df_avg')).otherwise(F.col('loser_df_avg'))) \
.withColumn('id_b_df_avg', F.when(F.col('random_flip')>= 0.5, F.col('loser_df_avg')).otherwise(F.col
('winner_df_avg'))) \
.withColumn('id_a_minutes_sum', F.when(F.col('random_flip')>= 0.5, F.col('winner_minutes_sum')).otherwise(F.col('loser_minutes_sum'))) \
.withColumn('id_b_minutes_sum', F.when(F.col('random_flip')>= 0.5, F.col('loser_minutes_sum')).otherwise(F.col('winner_minutes_sum'))) \
.withColumn('minutes_diff', F.col('id_a_minutes_sum') - F.col('id_b_minutes_sum')) \
.withColumn("round_encoded",
        F.when(F.col('round') == "RR", 0)
        .when(F.col('round') == "R128", 1)
        .when(F.col('round') == "R64", 2)
        .when(F.col('round') == "R32", 3)
        .when(F.col('round') == "R16", 4)
        .when(F.col('round') == "QF", 5)
        .when(F.col('round') == "SF", 6)
        .when(F.col('round') == "BR", 6.5)
        .when(F.col('round') == "F", 7)
        .otherwise(None)) \
.withColumn('id_a_won', F.when(F.col('random_flip')>= 0.5, 1).otherwise(0)) \
.drop('winner_hand', 'loser_hand', 'winner_age', 'loser_age', 'winner_rank', 'loser_rank', 'random_flip',
      'winner_first_pct_avg', 'loser_first_pct_avg', 'winner_second_pct_avg', 'loser_second_pct_avg',
      'winner_bp_saved_pct_avg', 'loser_bp_saved_pct_avg', 'winner_bp_convert_pct_avg', 'loser_bp_convert_pct_avg',
      'winner_return_pt_win_pct_avg', 'loser_return_pt_win_pct_avg', 'winner_break_pct_avg', 'loser_break_pct_avg',
      'winner_df_avg', 'loser_df_avg', 'winner_minutes_sum', 'loser_minutes_sum', 'winner_first_win_pct_avg',
      'loser_first_win_pct_avg', 'winner_second_win_pct_avg', 'loser_second_win_pct_avg', 'winner_id', 'loser_id',
      'id', 'round', 'id_a', 'id_b') \
.filter(F.col('surface').isNotNull() & F.col('rank_diff').isNotNull()
        & F.col('id_a_age').isNotNull() & F.col('id_b_age').isNotNull()
        & F.col('id_a_first_pct_avg').isNotNull() & F.col('id_b_first_pct_avg').isNotNull()
        & F.col('id_a_first_win_pct_avg').isNotNull() & F.col('id_b_first_win_pct_avg').isNotNull()
        & F.col('id_a_second_win_pct_avg').isNotNull() & F.col('id_b_second_win_pct_avg').isNotNull()
        & F.col('id_a_bp_saved_pct_avg').isNotNull() & F.col('id_b_bp_saved_pct_avg').isNotNull()
        & F.col('id_a_bp_convert_pct_avg').isNotNull() & F.col('id_b_bp_convert_pct_avg').isNotNull()
        & F.col('id_a_return_pt_win_pct_avg').isNotNull() & F.col('id_b_return_pt_win_pct_avg').isNotNull()
        & F.col('id_a_break_pct_avg').isNotNull() & F.col('id_b_break_pct_avg').isNotNull()
        & F.col('id_a_df_avg').isNotNull() & F.col('id_b_df_avg').isNotNull()
        & F.col('minutes_diff').isNotNull() & F.col('id_a_minutes_sum').isNotNull()
        & F.col('id_b_minutes_sum').isNotNull())

display(d)

#MLFlow

Because the number of models is about to increase substantially, whether that be from going from v1 to v2 or changing the rolling average window, I am now going to implement MLFlow to keep track of all the models.

In [0]:
mlflow.set_tracking_uri("databricks")

experiment_path = "/Users/mrfrankhutch04@gmail.com/tennis-match-prediction"
experiment = mlflow.get_experiment_by_name(experiment_path)

if experiment is not None:
    experiment_id = experiment.experiment_id
else:
    experiment_id = mlflow.create_experiment(
        name=experiment_path
    )

mlflow.set_experiment(experiment_path)

In [0]:
#Retroactively add v1 models
with mlflow.start_run(run_name='v1-log-reg'):
    mlflow.log_param('model_version', 'v1')
    mlflow.log_param('model_type', 'logistic regression')
    mlflow.log_param('features', 'surface, rank, round, hand, age, tourney_level')
    mlflow.log_metric('auc_roc', .6698)
    mlflow.log_metric('auc_pr', .6824)
    mlflow.log_metric('accuracy', .6178)
    mlflow.log_metric('precision_weighted', .6213)
    mlflow.log_metric('recall_weighted', .6178)
    mlflow.log_metric('f1_weighted', .6175)
    mlflow.log_metric('true_positives', 1386)
    mlflow.log_metric('true_negatives', 1436)
    mlflow.log_metric('false_positives', 743)
    mlflow.log_metric('false_negatives', 1003)

with mlflow.start_run(run_name='v1-random-forest'):
    mlflow.log_param('model_version', 'v1')
    mlflow.log_param('model_type', 'random forest')
    mlflow.log_param('features', 'surface, rank, round, hand, age, tourney_level')
    mlflow.log_param('numTrees', 50)
    mlflow.log_param('maxDepth', 10)
    mlflow.log_metric('auc_roc', .6953)
    mlflow.log_metric('auc_pr', .7056)
    mlflow.log_metric('accuracy', .6283)
    mlflow.log_metric('precision_weighted', .6299)
    mlflow.log_metric('recall_weighted', .6283)
    mlflow.log_metric('f1_weighted', .6285)
    mlflow.log_metric('true_positives', 1467)
    mlflow.log_metric('true_negatives', 1403)
    mlflow.log_metric('false_positives', 776)
    mlflow.log_metric('false_negatives', 922)

with mlflow.start_run(run_name='v1-gradient-boosting'):
    mlflow.log_param('model_version', 'v1')
    mlflow.log_param('model_type', 'gradient boosting')
    mlflow.log_param('features', 'surface, rank, round, hand, age, tourney_level')
    mlflow.log_param('maxDepth', 6)
    mlflow.log_param('maxIter', 20)
    mlflow.log_metric('auc_roc', .6967)
    mlflow.log_metric('auc_pr', .7066)
    mlflow.log_metric('accuracy', .6338)
    mlflow.log_metric('precision_weighted', .6358)
    mlflow.log_metric('recall_weighted', .6338)
    mlflow.log_metric('f1_weighted', .6339)
    mlflow.log_metric('true_positives', 1466)
    mlflow.log_metric('true_negatives', 1429)
    mlflow.log_metric('false_positives', 750)
    mlflow.log_metric('false_negatives', 923)


I originally tried to create v2 models using Pyspark.ml, but the models kept reaching the file limit of my free Databricks account. This led me to download my prepped data and access it on my jupyter notebook. I then created the models using sklearn. Because of less limitations, I decided to change Gradient Boosting to XGBoost for v2. I have posted the following lines of code that appear in my jupyter notebook as well as my logging of all the experiments on this notebook. The v1 models were also replicated to confirm the processes were the same.

In [0]:
from sklearn.metrics import (roc_auc_score, average_precision_score, accuracy_score,
    precision_score, recall_score, f1_score, confusion_matrix)

def evaluate_model(y_test, y_pred, y_prob):
    # ==========================================
    # 1. AUC Metrics (BinaryClassificationEvaluator)
    # ==========================================
    auc_roc = roc_auc_score(y_test, y_prob)
    auc_pr = average_precision_score(y_test, y_prob)

    # ==========================================
    # 2. Accuracy, Precision, Recall, F1 (MulticlassClassificationEvaluator)
    # ==========================================
    accuracy = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred, average='weighted', zero_division=0)
    recall = recall_score(y_test, y_pred, average='weighted', zero_division=0)
    f1 = f1_score(y_test, y_pred, average='weighted', zero_division=0)

    # ==========================================
    # 3. Confusion Matrix (manual calculation)
    # ==========================================
    tn, fp, fn, tp = confusion_matrix(y_test, y_pred).ravel()

    print("=" * 60)
    print(f"EVALUATION METRICS")
    print("=" * 60)
    print(f"AUC-ROC:              {auc_roc:.4f}")
    print(f"AUC-PR:               {auc_pr:.4f}")
    print(f"Accuracy:             {accuracy:.4f}")
    print(f"Precision (Weighted): {precision:.4f}")
    print(f"Recall (Weighted):    {recall:.4f}")
    print(f"F1 Score:             {f1:.4f}")
    print("=" * 60)
    print("CONFUSION MATRIX")
    print("=" * 60)
    print(f"True Positives:       {tp:,}")
    print(f"True Negatives:       {tn:,}")
    print(f"False Positives:      {fp:,}")
    print(f"False Negatives:      {fn:,}")
    print("=" * 60)

    return {
        'auc_roc': auc_roc,
        'auc_pr': auc_pr,
        'accuracy': accuracy,
        'precision_weighted': precision,
        'recall_weighted': recall,
        'f1_weighted': f1,
        'confusion_matrix': {'tp': int(tp), 'tn': int(tn), 'fp': int(fp), 'fn': int(fn)}
    }

In [0]:
#v2
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OrdinalEncoder, OneHotEncoder
from sklearn.compose import ColumnTransformer

#Traininng and testing data
y=d['id_a_won']

#X variables
feature_cols = ['surface', 'tourney_level', 'round_encoded', 'best_of', 'id_a_rank',
                'id_b_rank','id_a_age','id_b_age','rank_diff','id_a_hand','id_b_hand',
                'id_a_first_pct_avg', 'id_b_first_pct_avg', 'id_a_first_win_pct_avg',
                'id_b_first_win_pct_avg', 'id_a_second_win_pct_avg', 'id_b_second_win_pct_avg',
                'id_a_bp_saved_pct_avg', 'id_b_bp_saved_pct_avg', 'id_a_bp_convert_pct_avg',
                'id_b_bp_convert_pct_avg', 'id_a_return_pt_win_pct_avg', 'id_b_return_pt_win_pct_avg',
                'id_a_break_pct_avg', 'id_b_break_pct_avg', 'id_a_df_avg', 'id_b_df_avg',
                'id_a_minutes_sum', 'id_b_minutes_sum', 'minutes_diff']

X=d[feature_cols]

x_train, x_test, y_train, y_test = train_test_split(X, y, train_size=.8, test_size=.2, shuffle=False, random_state=42)

In [0]:
#v2 Logistic Regression
from sklearn.linear_model import LogisticRegression
    
#Preprocessor
categorical_cols = ["surface", "tourney_level"]

categorical_transformer = Pipeline(steps=[
    ("ordinal", OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)),
    ("onehot",  OneHotEncoder(handle_unknown="ignore"))
])

numeric_cols = ['round_encoded', 'best_of', 'id_a_rank',
                'id_b_rank','id_a_age','id_b_age','rank_diff','id_a_hand','id_b_hand',
                'id_a_first_pct_avg', 'id_b_first_pct_avg', 'id_a_first_win_pct_avg',
                'id_b_first_win_pct_avg', 'id_a_second_win_pct_avg', 'id_b_second_win_pct_avg',
                'id_a_bp_saved_pct_avg', 'id_b_bp_saved_pct_avg', 'id_a_bp_convert_pct_avg',
                'id_b_bp_convert_pct_avg', 'id_a_return_pt_win_pct_avg', 'id_b_return_pt_win_pct_avg',
                'id_a_break_pct_avg', 'id_b_break_pct_avg', 'id_a_df_avg', 'id_b_df_avg',
                'id_a_minutes_sum', 'id_b_minutes_sum', 'minutes_diff']

preprocessor = ColumnTransformer(
    transformers=[
        ("cat", categorical_transformer, categorical_cols),
        ("num", "passthrough", numeric_cols)
    ]
)
 
#Model
lr = LogisticRegression(solver="lbfgs", max_iter=500, n_jobs=-1, random_state=42)

#Pipeline
pipeline_lr = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model', lr)
])

#Train
lr_model = pipeline_lr.fit(x_train, y_train)

#Save
local_path = "v2-log-reg-5-match-ma-8-years.pkl"
joblib.dump(lr_model, local_path)
s3.upload_file(
    Filename="v2-log-reg-5-match-ma-8-years.pkl",
    Bucket="data-storage-for-projects",
    Key="Tennis Analytics Project/v2/v2-models/logistic-regressions/v2-log-reg-5-match-ma-8-years.pkl"
)

os.remove(local_path)
print("Local file removed.")

#Predict
lr_pred = lr_model.predict(x_test)
lr_prob = lr_model.predict_proba(x_test)[:, 1]

#Evaluate
model_eval = evaluate_model(y_test, lr_pred, lr_prob)

model_metrics = {'run_path': local_path[:-4],
'model_version': 'v2',
'model_type': 'logistic regression',
'features': feature_cols,
'auc_roc': model_eval['auc_roc'],
'auc_pr': model_eval['auc_pr'],
'accuracy': model_eval['accuracy'],
'precision_weighted': model_eval['precision_weighted'],
'recall_weighted': model_eval['recall_weighted'],
'f1_weighted': model_eval['f1_weighted'],
'true_positives': model_eval['confusion_matrix']['tp'],
'true_negatives': model_eval['confusion_matrix']['tn'],
'false_positives': model_eval['confusion_matrix']['fp'],
'false_negatives': model_eval['confusion_matrix']['fn']
}

# Feature Importance
feature_names = lr_model.named_steps['preprocessor'].get_feature_names_out()
coeffs = lr_model.named_steps['model'].coef_.flatten()

feature_importance = pd.DataFrame({
    'feature': feature_names,
    'importance': coeffs
}).sort_values('importance', ascending=False)

print(feature_importance)
print(model_metrics)

#Remove from Databricks Memory to create more models
del lr_model, lr_pred, lr_prob

In [0]:
#v2 Logistic Regression Model
metrics = {'run_path': 'v2-log-reg-5-match-ma-4-years', 'model_version': 'v2', 'model_type': 'logistic regression', 'features': ['surface', 'tourney_level', 'round_encoded', 'best_of', 'id_a_rank', 'id_b_rank', 'id_a_age', 'id_b_age', 'rank_diff', 'id_a_hand', 'id_b_hand', 'id_a_first_pct_avg', 'id_b_first_pct_avg', 'id_a_first_win_pct_avg', 'id_b_first_win_pct_avg', 'id_a_second_win_pct_avg', 'id_b_second_win_pct_avg', 'id_a_bp_saved_pct_avg', 'id_b_bp_saved_pct_avg', 'id_a_bp_convert_pct_avg', 'id_b_bp_convert_pct_avg', 'id_a_return_pt_win_pct_avg', 'id_b_return_pt_win_pct_avg', 'id_a_break_pct_avg', 'id_b_break_pct_avg', 'id_a_df_avg', 'id_b_df_avg', 'id_a_minutes_sum', 'id_b_minutes_sum', 'minutes_diff'], 'auc_roc': 0.6507113397906954, 'auc_pr': 0.6374943345468282, 'accuracy': 0.6124599175446633, 'precision_weighted': 0.612400441172934, 'recall_weighted': 0.6124599175446633, 'f1_weighted': 0.6120264625161669, 'true_positives': 716, 'true_negatives': 621, 'false_positives': 451, 'false_negatives': 395}

with mlflow.start_run(run_name=metrics['run_path']):
    mlflow.log_param('model_version', metrics['model_version'])
    mlflow.log_param('model_type', metrics['model_type'])
    mlflow.log_param('features', metrics['features'])
    mlflow.log_metric('auc_roc', metrics['auc_roc'])
    mlflow.log_metric('auc_pr', metrics['auc_pr'])
    mlflow.log_metric('accuracy', metrics['accuracy'])
    mlflow.log_metric('precision_weighted', metrics['precision_weighted'])
    mlflow.log_metric('recall_weighted', metrics['recall_weighted'])
    mlflow.log_metric('f1_weighted', metrics['f1_weighted'])
    mlflow.log_metric('true_positives', metrics['true_positives'])
    mlflow.log_metric('true_negatives', metrics['true_negatives'])
    mlflow.log_metric('false_positives', metrics['false_positives'])
    mlflow.log_metric('false_negatives', metrics['false_negatives'])

In [0]:
#v2 Random Forest
from sklearn.ensemble import RandomForestClassifier

#Preprocessor
categorical_cols = ["surface", "tourney_level"]

categorical_transformer = Pipeline(steps=[
    ("ordinal", OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)),
    ("onehot",  OneHotEncoder(handle_unknown="ignore"))
])

numeric_cols = ['round_encoded', 'best_of', 'id_a_rank',
                'id_b_rank','id_a_age','id_b_age','rank_diff','id_a_hand','id_b_hand',
                'id_a_first_pct_avg', 'id_b_first_pct_avg', 'id_a_first_win_pct_avg',
                'id_b_first_win_pct_avg', 'id_a_second_win_pct_avg', 'id_b_second_win_pct_avg',
                'id_a_bp_saved_pct_avg', 'id_b_bp_saved_pct_avg', 'id_a_bp_convert_pct_avg',
                'id_b_bp_convert_pct_avg', 'id_a_return_pt_win_pct_avg', 'id_b_return_pt_win_pct_avg',
                'id_a_break_pct_avg', 'id_b_break_pct_avg', 'id_a_df_avg', 'id_b_df_avg',
                'id_a_minutes_sum', 'id_b_minutes_sum', 'minutes_diff']

preprocessor = ColumnTransformer(
    transformers=[
        ("cat", categorical_transformer, categorical_cols),
        ("num", "passthrough", numeric_cols)
    ]
)
 
#Model
rf = RandomForestClassifier(n_estimators=50, max_depth=20, random_state=42)

#Pipeline
pipeline_rf = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model', rf)
])

#Train
rf_model = pipeline_rf.fit(x_train, y_train)

#Save
local_path = "v2-rand-forest-5-match-ma-8-years.pkl"
joblib.dump(rf_model, local_path)
s3.upload_file(
    Filename="v2-rand-forest-5-match-ma-8-years.pkl",
    Bucket="data-storage-for-projects",
    Key="Tennis Analytics Project/v2/v2-models/random-forests/v2-rand-forest-5-match-ma-8-years.pkl"
)

os.remove(local_path)
print("Local file removed.")

#Predict
rf_pred = rf_model.predict(x_test)
rf_prob = rf_model.predict_proba(x_test)[:, 1]

#Evaluate
model_eval = evaluate_model(y_test, rf_pred, rf_prob)

model_metrics = {'run_path': local_path[:-4],
'model_version': 'v2',
'model_type': 'random forest',
'features': feature_cols,
'auc_roc': model_eval['auc_roc'],
'auc_pr': model_eval['auc_pr'],
'accuracy': model_eval['accuracy'],
'precision_weighted': model_eval['precision_weighted'],
'recall_weighted': model_eval['recall_weighted'],
'f1_weighted': model_eval['f1_weighted'],
'true_positives': model_eval['confusion_matrix']['tp'],
'true_negatives': model_eval['confusion_matrix']['tn'],
'false_positives': model_eval['confusion_matrix']['fp'],
'false_negatives': model_eval['confusion_matrix']['fn']
}

# Feature Importance
feature_names = rf_model.named_steps['preprocessor'].get_feature_names_out()
coeffs = rf_model.named_steps['model'].feature_importances_

feature_importance = pd.DataFrame({
    'feature': feature_names,
    'importance': coeffs
}).sort_values('importance', ascending=False)

print(feature_importance)
print(model_metrics)

#Remove from Databricks Memory to create more models
del rf_model, rf_pred, rf_prob

In [0]:
#v2 Random Forest
metrics = {'run_path': 'v2-rand-forest-5-match-ma-4-years', 'model_version': 'v2', 'model_type': 'random forest', 'features': ['surface', 'tourney_level', 'round_encoded', 'best_of', 'id_a_rank', 'id_b_rank', 'id_a_age', 'id_b_age', 'rank_diff', 'id_a_hand', 'id_b_hand', 'id_a_first_pct_avg', 'id_b_first_pct_avg', 'id_a_first_win_pct_avg', 'id_b_first_win_pct_avg', 'id_a_second_win_pct_avg', 'id_b_second_win_pct_avg', 'id_a_bp_saved_pct_avg', 'id_b_bp_saved_pct_avg', 'id_a_bp_convert_pct_avg', 'id_b_bp_convert_pct_avg', 'id_a_return_pt_win_pct_avg', 'id_b_return_pt_win_pct_avg', 'id_a_break_pct_avg', 'id_b_break_pct_avg', 'id_a_df_avg', 'id_b_df_avg', 'id_a_minutes_sum', 'id_b_minutes_sum', 'minutes_diff'], 'auc_roc': 0.6798714852828567, 'auc_pr': 0.6791047049273793, 'accuracy': 0.63078332569858, 'precision_weighted': 0.6308716561673112, 'recall_weighted': 0.63078332569858, 'f1_weighted': 0.6308084321472727, 'true_positives': 702, 'true_negatives': 675, 'false_positives': 397, 'false_negatives': 409}

with mlflow.start_run(run_name=metrics['run_path']):
    mlflow.log_param('model_version', metrics['model_version'])
    mlflow.log_param('model_type', metrics['model_type'])
    mlflow.log_param('features', metrics['features'])
    mlflow.log_metric('auc_roc', metrics['auc_roc'])
    mlflow.log_metric('auc_pr', metrics['auc_pr'])
    mlflow.log_metric('accuracy', metrics['accuracy'])
    mlflow.log_metric('precision_weighted', metrics['precision_weighted'])
    mlflow.log_metric('recall_weighted', metrics['recall_weighted'])
    mlflow.log_metric('f1_weighted', metrics['f1_weighted'])
    mlflow.log_metric('true_positives', metrics['true_positives'])
    mlflow.log_metric('true_negatives', metrics['true_negatives'])
    mlflow.log_metric('false_positives', metrics['false_positives'])
    mlflow.log_metric('false_negatives', metrics['false_negatives'])

In [0]:
#v2 XGBoost
from xgboost import XGBClassifier

#Preprocessor
categorical_cols = ["surface", "tourney_level"]

categorical_transformer = Pipeline(steps=[
    ("ordinal", OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)),
    ("onehot",  OneHotEncoder(handle_unknown="ignore"))
])

numeric_cols = ['round_encoded', 'best_of', 'id_a_rank',
                'id_b_rank','id_a_age','id_b_age','rank_diff','id_a_hand','id_b_hand',
                'id_a_first_pct_avg', 'id_b_first_pct_avg', 'id_a_first_win_pct_avg',
                'id_b_first_win_pct_avg', 'id_a_second_win_pct_avg', 'id_b_second_win_pct_avg',
                'id_a_bp_saved_pct_avg', 'id_b_bp_saved_pct_avg', 'id_a_bp_convert_pct_avg',
                'id_b_bp_convert_pct_avg', 'id_a_return_pt_win_pct_avg', 'id_b_return_pt_win_pct_avg',
                'id_a_break_pct_avg', 'id_b_break_pct_avg', 'id_a_df_avg', 'id_b_df_avg',
                'id_a_minutes_sum', 'id_b_minutes_sum', 'minutes_diff']

preprocessor = ColumnTransformer(
    transformers=[
        ("cat", categorical_transformer, categorical_cols),
        ("num", "passthrough", numeric_cols)
    ]
)
 
#Model
xg = XGBClassifier(n_estimators=300, max_depth=6, learning_rate=0.05, subsample=0.9,
    colsample_bytree=0.9, eval_metric="logloss", random_state=42)

#Pipeline
pipeline_xg = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model', xg)
])

#Train
xg_model = pipeline_xg.fit(x_train, y_train)

#Save
local_path = "v2-xgboost-5-match-ma-8-years.bin"
joblib.dump(xg_model, local_path)
s3.upload_file(
    Filename="v2-xgboost-5-match-ma-8-years.bin",
    Bucket="data-storage-for-projects",
    Key="Tennis Analytics Project/v2/v2-models/xgboosts/v2-xgboost-5-match-ma-8-years.bin"
)

os.remove(local_path)
print("Local file removed.")

#Predict
xg_pred = xg_model.predict(x_test)
xg_prob = xg_model.predict_proba(x_test)[:, 1]

#Evaluate
model_eval = evaluate_model(y_test, xg_pred, xg_prob)

model_metrics = {'run_path': local_path[:-4],
'model_version': 'v2',
'model_type': 'xgboost',
'features': feature_cols,
'auc_roc': model_eval['auc_roc'],
'auc_pr': model_eval['auc_pr'],
'accuracy': model_eval['accuracy'],
'precision_weighted': model_eval['precision_weighted'],
'recall_weighted': model_eval['recall_weighted'],
'f1_weighted': model_eval['f1_weighted'],
'true_positives': model_eval['confusion_matrix']['tp'],
'true_negatives': model_eval['confusion_matrix']['tn'],
'false_positives': model_eval['confusion_matrix']['fp'],
'false_negatives': model_eval['confusion_matrix']['fn']
}

# Feature Importance
feature_names = xg_model.named_steps['preprocessor'].get_feature_names_out()
coeffs = xg_model.named_steps['model'].feature_importances_

feature_importance = pd.DataFrame({
    'feature': feature_names,
    'importance': coeffs
}).sort_values('importance', ascending=False)

print(feature_importance)
print(model_metrics)

#Remove from Databricks Memory to create more models
del xg_model, xg_pred, xg_prob

In [0]:
#v2 XGBoost
metrics = {'run_path': 'v2-xgboost-5-match-ma-4-years', 'model_version': 'v2', 'model_type': 'xgboost', 'features': ['surface', 'tourney_level', 'round_encoded', 'best_of', 'id_a_rank', 'id_b_rank', 'id_a_age', 'id_b_age', 'rank_diff', 'id_a_hand', 'id_b_hand', 'id_a_first_pct_avg', 'id_b_first_pct_avg', 'id_a_first_win_pct_avg', 'id_b_first_win_pct_avg', 'id_a_second_win_pct_avg', 'id_b_second_win_pct_avg', 'id_a_bp_saved_pct_avg', 'id_b_bp_saved_pct_avg', 'id_a_bp_convert_pct_avg', 'id_b_bp_convert_pct_avg', 'id_a_return_pt_win_pct_avg', 'id_b_return_pt_win_pct_avg', 'id_a_break_pct_avg', 'id_b_break_pct_avg', 'id_a_df_avg', 'id_b_df_avg', 'id_a_minutes_sum', 'id_b_minutes_sum', 'minutes_diff'], 'auc_roc': 0.6780700458105511, 'auc_pr': 0.6799282881813442, 'accuracy': 0.6220797068254695, 'precision_weighted': 0.6220859967807384, 'recall_weighted': 0.6220797068254695, 'f1_weighted': 0.6220827212745649, 'true_positives': 698, 'true_negatives': 660, 'false_positives': 412, 'false_negatives': 413}

with mlflow.start_run(run_name=metrics['run_path']):
    mlflow.log_param('model_version', metrics['model_version'])
    mlflow.log_param('model_type', metrics['model_type'])
    mlflow.log_param('features', metrics['features'])
    mlflow.log_metric('auc_roc', metrics['auc_roc'])
    mlflow.log_metric('auc_pr', metrics['auc_pr'])
    mlflow.log_metric('accuracy', metrics['accuracy'])
    mlflow.log_metric('precision_weighted', metrics['precision_weighted'])
    mlflow.log_metric('recall_weighted', metrics['recall_weighted'])
    mlflow.log_metric('f1_weighted', metrics['f1_weighted'])
    mlflow.log_metric('true_positives', metrics['true_positives'])
    mlflow.log_metric('true_negatives', metrics['true_negatives'])
    mlflow.log_metric('false_positives', metrics['false_positives'])
    mlflow.log_metric('false_negatives', metrics['false_negatives'])

The next phase of v2 will be to change the different parameters to see if the models improve. The player moving average stats, the number of years of data, and model specific parameters will be altered and logged.